# Preprocessing transcriptome data

This notebook covers the first part of the Eykthyr pipeline: loading spatial
RNA-seq data, performing standard quality-control and normalization steps, and
learning spatially-regularized metagene representations via
[Popari](https://github.com/raphael-group/popari).

**Inputs required:**
- `data/mouse_embryo2_rna.h5ad` — AnnData with raw counts in `.X` and spatial
  coordinates in `.obsm['spatial']`.

**Output:**
- `spatialatacrna.h5ad` — directory containing preprocessed RNA, Popari model,
  and metagene embeddings, ready for the training notebook.

In [ ]:
from eykthyr.eykthyr import Eykthyr, load_anndata
import scanpy as sc

In [ ]:
e = Eykthyr()

## Step 1 — Load data

Initialize an empty `Eykthyr` object and load the spatial RNA AnnData.
The AnnData must contain raw counts in `.X` and 2-D tissue coordinates in
`.obsm['spatial']`.

In [ ]:
adrna = sc.read('data/mouse_embryo2_rna.h5ad')
adrna

In [ ]:
e.set_RNA([adrna])

In [ ]:
e.preprocess_rna(make_plots=True)

## Step 2 — Preprocess RNA

`preprocess_rna` applies the following steps to each dataset:

1. Filter genes expressed in fewer than 5 cells.
2. Filter cells with fewer than 10 total counts.
3. Normalize total counts to 10,000 per cell.
4. Log1p-transform.
5. (Optional) PCA → neighbors → UMAP → Leiden clustering for a QC plot.

Raw counts are preserved in `.layers['raw']`.

In [ ]:
e.compute_metagenes()

## Step 3 — Compute metagenes

`compute_metagenes` fits the [Popari](https://github.com/raphael-group/popari)
spatially-regularized NMF model to learn `K` metagenes.

Key parameters:
- `K` — number of metagenes (default 16). A larger `K` captures finer programs
  but requires more computation.
- `initial_iterations` — iterations without spatial regularization (warm-up).
- `spatial_iterations` — iterations with spatial affinities (main training).

This step requires a CUDA-capable GPU by default (`torch_context=dict(device='cuda:0', ...)`).
For CPU-only use, pass `torch_context=dict(device='cpu', dtype=torch.float64)`.

In [ ]:
e.analyze_metagenes()

## Step 4 — Analyze metagenes

`analyze_metagenes` post-processes the Popari embeddings:

- Normalizes metagene scores per cell.
- Runs Leiden clustering on the normalized metagene space.
- Computes a UMAP of the metagene space for visualization.

The resulting UMAP is colored by Leiden clusters and any additional
`cluster_annotation` keys passed to `preprocess_rna`.

In [ ]:
e.save_anndata('spatialatacrna.h5ad')

## Step 5 — Save session

Saves all AnnData objects and the Popari model to a directory named
`spatialatacrna/`. Pass the same path to `load_anndata()` to restore
the session in the training notebook.